# Databricks Lakehouse - Bronze Layer Ingestion (`01_generate_bronze`)

## Overview
This notebook simulates and ingests synthetic patient telemetry data into a **Delta Lake Bronze Table** (`workspace.neuro_insight.bronze_telemetry`). It follows best practices for data engineering in Databricks, including parameterization via Databricks Widgets, synthetic feature generation, quality issue simulation, and Delta table persistence.

---

### Step 1: Initialize Parameters & Environment
Define pipeline runtime parameters using Databricks Widgets and import necessary PySpark functions.

In [ ]:
from pyspark.sql import functions as F

# Create widget parameter for batch execution
dbutils.widgets.text("batch_id", "001", "Batch ID")

# Retrieve batch ID value
batch_id = dbutils.widgets.get("batch_id")
print("Current Batch ID:", batch_id)

### Step 2: Generate Synthetic Patient Telemetry Data
1. Define base patient and device identifiers.
2. Generate 100 sequential event records per patient using a cross-join operation.
3. Add synthetic metric columns (timestamps, tremor scores, reaction times, gait speed, cognitive scores, and heart rates).

In [ ]:
# Create base DataFrame of patients and assigned devices
patients = [
    ("P001", "D001"),
    ("P002", "D002"),
    ("P003", "D003"),
    ("P004", "D004"),
    ("P005", "D005")
]
patient_df = spark.createDataFrame(patients, ["patient_id", "device_id"])

# Create 100 sequential event IDs
events = spark.range(100).withColumnRenamed("id", "event_id")

# Cross-join patients and events (5 patients * 100 events = 500 records)
telemetry_df = patient_df.crossJoin(events)

print("Number of base records generated:", telemetry_df.count())

### Step 3: Populate Synthetic Telemetry Metrics
Use PySpark function expressions to compute realistic medical metrics and incremental timestamps.

In [ ]:
# Add metric columns with randomized realistic values
telemetry_df = (
    telemetry_df
    .withColumn(
        "event_timestamp",
        F.current_timestamp() - F.expr("INTERVAL 1 DAY") + F.expr("INTERVAL 1 SECOND") * F.col("event_id")
    )
    .withColumn("tremor_score", F.round(F.rand(seed=42) * 10, 2))
    .withColumn("reaction_time_ms", F.round(300 + F.rand(seed=43) * 500, 0))
    .withColumn("gait_speed_mps", F.round(0.5 + F.rand(seed=44) * 1.0, 2))
    .withColumn("cognitive_score", F.round(50 + F.rand(seed=45) * 50, 0))
    .withColumn("heart_rate_bpm", F.round(60 + F.rand(seed=46) * 40, 0))
)

### Step 4: Inject Bad / Malformed Data
In a real-world pipeline, Bronze tables store raw, uncleaned data (including missing values or corrupt timestamps). Here we append sample bad records to test downstream Silver-layer data cleansing logic.

In [ ]:
# Define malformed data records (e.g., missing timestamps or invalid out-of-bound values)
bad_data = [
    ("P006", "D006", 999, None, None, 450.0, 1.0, 75.0, 70.0),
    ("P007", "D007", 1000, None, 12.5, 500.0, 1.1, 80.0, 72.0)
]

bad_df = spark.createDataFrame(
    bad_data,
    "patient_id STRING, device_id STRING, event_id BIGINT, "
    "event_timestamp TIMESTAMP, tremor_score DOUBLE, reaction_time_ms DOUBLE, "
    "gait_speed_mps DOUBLE, cognitive_score DOUBLE, heart_rate_bpm DOUBLE"
)

# Combine clean synthetic data with bad data
raw_df = telemetry_df.unionByName(bad_df)

print("Total raw records (including anomalies):", raw_df.count())

### Step 5: Write Raw Telemetry to Delta Lake (Bronze Table)
Persist the aggregated raw dataset into Unity Catalog / Databricks metastore under table `workspace.neuro_insight.bronze_telemetry` using Delta format.

In [ ]:
(
    raw_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.neuro_insight.bronze_telemetry")
)

print("Bronze table workspace.neuro_insight.bronze_telemetry created successfully.")